# RF 03 — SHAP Error Analysis

**Thesis project:** Explainability-Guided Feature Analysis for Reducing False Positives in Phishing Email Detection Without Increasing False Negatives.

This notebook explains errors made by the **locked Random Forest baseline**. It covers the original Random Forest Steps **6–11**. It includes the original recorded results, clear tables, and the **Top 20 FP SHAP bar chart** from Step 9A.

**Simple idea:** SHAP helps show which words influenced the model's prediction. We study **62 false positives (FPs)** and **15 false negatives (FNs)**. The model is **not retrained** here.

**Evidence rule:** Tables marked *Recorded results* reproduce values you reported. Tables marked *Computed* are generated when the original SHAP calculation runs. The chart reproduces the original Step 9A visualization; no extra experimental figures are introduced.

## 1. Restore and check the official baseline

This notebook loads the saved **RF 02** baseline model, TF-IDF vectorizer, data matrices, labels, and predictions from Google Drive. It uses the exact filenames written by the GitHub RF 02 checkpoint export. It checks the confusion matrix and the original prediction SHA-256 before SHAP analysis.

**Before running:** Execute the RF 02 checkpoint-saving cell at least once, and keep the saved files in the official baseline folder. If your earlier original Step 5B files have different filenames, recover them using your original Step 5B code rather than renaming or guessing files. This notebook does **not** retrain the model or refit TF-IDF.

In [ ]:
import hashlib
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
from sklearn.metrics import confusion_matrix

# Run in Google Colab. Mount your own Drive when prompted.
from google.colab import drive
drive.mount('/content/drive')

BASELINE_DIR = Path('/content/drive/MyDrive/Random_Forest_SHAP_Guided_Phishing_Detection/Official_Random_Forest_Baseline')
assert BASELINE_DIR.is_dir(), f'Official baseline folder not found: {BASELINE_DIR}'
print('Official baseline directory:', BASELINE_DIR)
print('Available checkpoint files:', sorted(p.name for p in BASELINE_DIR.iterdir()))

In [ ]:
# Restore the official RF 02 checkpoint, using the exact filenames saved by
# rf_02_official_baseline_corrected.ipynb. No model retraining or TF-IDF refitting.
import json
from scipy import sparse

checkpoint_files = {
    'rf_baseline_model.joblib': 'model',
    'tfidf_vectorizer.joblib': 'vectorizer',
    'X_train_tfidf.npz': 'train matrix',
    'X_test_tfidf.npz': 'test matrix',
    'y_train.npy': 'training labels',
    'y_test.npy': 'test labels',
    'baseline_predictions.npy': 'baseline predictions',
    'baseline_probabilities.npy': 'baseline probabilities',
    'baseline_manifest.json': 'manifest',
}
missing_files = [name for name in checkpoint_files if not (BASELINE_DIR / name).is_file()]
if missing_files:
    raise FileNotFoundError(
        'The RF 02 checkpoint is missing these files: ' + ', '.join(missing_files) +
        '. Run the checkpoint export in RF 02, or use the original Step 5B '
        'recovery code if your existing Drive files have different names.'
    )

rf_model = joblib.load(BASELINE_DIR / 'rf_baseline_model.joblib')
tfidf_vectorizer = joblib.load(BASELINE_DIR / 'tfidf_vectorizer.joblib')
X_train_tfidf = sparse.load_npz(BASELINE_DIR / 'X_train_tfidf.npz')
X_test_tfidf = sparse.load_npz(BASELINE_DIR / 'X_test_tfidf.npz')
y_train = np.load(BASELINE_DIR / 'y_train.npy', allow_pickle=False).reshape(-1).astype(int)
y_test = np.load(BASELINE_DIR / 'y_test.npy', allow_pickle=False).reshape(-1).astype(int)
baseline_pred_rf = np.load(BASELINE_DIR / 'baseline_predictions.npy', allow_pickle=False).reshape(-1).astype(int)
baseline_prob_rf = np.load(BASELINE_DIR / 'baseline_probabilities.npy', allow_pickle=False).reshape(-1)
with open(BASELINE_DIR / 'baseline_manifest.json', encoding='utf-8') as f:
    baseline_manifest_rf = json.load(f)

assert X_train_tfidf.shape == (42861, 10000)
assert X_test_tfidf.shape == (10716, 10000)
assert len(y_train) == 42861 and len(y_test) == 10716
assert len(baseline_pred_rf) == 10716 and len(baseline_prob_rf) == 10716
assert len(rf_model.estimators_) == 100
assert np.array_equal(confusion_matrix(y_test, baseline_pred_rf, labels=[0, 1]), [[4686, 62], [15, 5953]])
assert baseline_manifest_rf['confusion_matrix_sklearn'] == [[4686, 62], [15, 5953]]
actual_hash = hashlib.sha256(np.ascontiguousarray(baseline_pred_rf).tobytes()).hexdigest()
assert actual_hash == baseline_manifest_rf['predictions_sha256']
assert actual_hash == '2c7f695b0dfae480ae17a5e59bb661b4e76606d45dc7e30bf2e8f0c74230cc2e'
false_positive_positions_rf = np.flatnonzero((y_test == 0) & (baseline_pred_rf == 1))
false_negative_positions_rf = np.flatnonzero((y_test == 1) & (baseline_pred_rf == 0))
assert len(false_positive_positions_rf) == 62
assert len(false_negative_positions_rf) == 15
print('Official RF baseline checkpoint verified: 62 FP and 15 FN.')
print('Prediction SHA-256:', actual_hash)


### Recorded baseline error counts

| Email type | Correct | Incorrect |
|---|---:|---:|
| Legitimate (class 0) | 4,686 | 62 false positives |
| Phishing (class 1) | 5,953 | 15 false negatives |

**Explanation:** An FP is a legitimate email wrongly marked as phishing. An FN is a phishing email wrongly marked as legitimate. These 77 errors are the focus of SHAP analysis.

## 2. SHAP setup and background (Steps 6–7)

The original run used **SHAP 0.52.0**, **NumPy 2.1.3**, and a fixed background of **100 training emails** (50 legitimate and 50 phishing). TreeExplainer used *interventional* explanations for the **phishing probability (class 1)**.

In [ ]:
# Use in the intended Colab environment, if SHAP is not installed:
# %pip install shap==0.52.0
import shap
print('SHAP:', shap.__version__, 'NumPy:', np.__version__)
assert shap.__version__ == '0.52.0', 'Use the recorded SHAP version for strict reproduction.'

rng = np.random.default_rng(42)
legit_idx = np.flatnonzero(y_train == 0)
phish_idx = np.flatnonzero(y_train == 1)
background_indices = np.concatenate([rng.choice(legit_idx, 50, replace=False), rng.choice(phish_idx, 50, replace=False)])
X_background_step8 = X_train_tfidf[background_indices].toarray()
assert X_background_step8.shape == (100, 10000)

explainer_rf = shap.TreeExplainer(rf_model, data=X_background_step8, feature_perturbation='interventional', model_output='probability')
print('Expected values:', explainer_rf.expected_value)

## 3. Explain all 62 false positives (Step 8)

For each FP, SHAP assigns a value to each TF-IDF feature. A positive value pushes the predicted phishing probability up. A negative value pushes it down. We process all 62 cases in small batches.

In [ ]:
def phishing_class_shap(values):
    if isinstance(values, list) and len(values) == 2:
        return np.asarray(values[1])
    arr = np.asarray(values)
    if arr.ndim == 3 and arr.shape[-1] == 2:
        return arr[:, :, 1]
    if arr.ndim == 2:
        return arr
    raise ValueError(f'Unexpected SHAP output shape: {arr.shape}')

fp_tfidf_rf = X_test_tfidf[false_positive_positions_rf]
fp_parts = []
for start in range(0, len(false_positive_positions_rf), 5):
    dense = fp_tfidf_rf[start:start+5].toarray()
    fp_parts.append(phishing_class_shap(explainer_rf.shap_values(dense, check_additivity=False)))
fp_shap_values_rf = np.vstack(fp_parts)
assert fp_shap_values_rf.shape == (62, 10000)
expected_rf = float(np.asarray(explainer_rf.expected_value).reshape(-1)[-1])
fp_reconstructed = expected_rf + fp_shap_values_rf.sum(axis=1)
fp_additivity_error = np.abs(fp_reconstructed - baseline_prob_rf[false_positive_positions_rf])
assert fp_additivity_error.max() < 1e-5
print('FP count:', len(false_positive_positions_rf))
print('Maximum additivity error:', fp_additivity_error.max())

### Recorded Step 8 verification table

| Check | Original result |
|---|---:|
| False positives explained | 62 |
| TF-IDF features | 10,000 |
| Background training emails | 100 |
| Background expected phishing probability | 0.5045715360018757 |
| Maximum additivity error | 3.378589696900747e-08 |
| Mean additivity error | 7.112174430035806e-09 |

**Explanation:** The very small additivity error shows that the SHAP contributions closely reconstruct the Random Forest prediction probability. These numbers are **recorded from your original experiment**, not generated outputs saved in this notebook.

## 4. Top 20 FP features: table and original bar chart (Steps 9 and 9A)

**Mean absolute SHAP** shows how strongly a word influenced the predictions, without saying whether the word pushed toward phishing or legitimate. The following table uses the **20 original values** you reported.

In [ ]:
# Recorded Step 9 table (values copied from the original RF output).
import pandas as pd
import matplotlib.pyplot as plt

recorded_fp_top20 = pd.DataFrame(
    [{'Rank': 1, 'Feature': 'wrote', 'Mean absolute SHAP': 0.0276509}, {'Rank': 2, 'Feature': 'org', 'Mean absolute SHAP': 0.02296829}, {'Rank': 3, 'Feature': 'code', 'Mean absolute SHAP': 0.0202585}, {'Rank': 4, 'Feature': 'escapenumber escapenumber', 'Mean absolute SHAP': 0.01361738}, {'Rank': 5, 'Feature': 'perl', 'Mean absolute SHAP': 0.00970759}, {'Rank': 6, 'Feature': 'list', 'Mean absolute SHAP': 0.00938977}, {'Rank': 7, 'Feature': 'mailing list', 'Mean absolute SHAP': 0.00732118}, {'Rank': 8, 'Feature': 'mailing', 'Mean absolute SHAP': 0.00687836}, {'Rank': 9, 'Feature': 'mailman listinfo', 'Mean absolute SHAP': 0.00665439}, {'Rank': 10, 'Feature': 'escapenumber', 'Mean absolute SHAP': 0.00663773}, {'Rank': 11, 'Feature': 'mailman', 'Mean absolute SHAP': 0.00655519}, {'Rank': 12, 'Feature': 'samba', 'Mean absolute SHAP': 0.00649657}, {'Rank': 13, 'Feature': 'listinfo', 'Mean absolute SHAP': 0.00622608}, {'Rank': 14, 'Feature': 'file', 'Mean absolute SHAP': 0.00599473}, {'Rank': 15, 'Feature': 'thanks', 'Mean absolute SHAP': 0.00587794}, {'Rank': 16, 'Feature': 'parrot', 'Mean absolute SHAP': 0.00478956}, {'Rank': 17, 'Feature': 'string', 'Mean absolute SHAP': 0.00438024}, {'Rank': 18, 'Feature': 'hi', 'Mean absolute SHAP': 0.00436996}, {'Rank': 19, 'Feature': 'use', 'Mean absolute SHAP': 0.00408579}, {'Rank': 20, 'Feature': 'rescapenumber', 'Mean absolute SHAP': 0.00402202}]
)
display(recorded_fp_top20.style.format({'Mean absolute SHAP': '{:.8f}'}))

### Figure 1 — Top 20 SHAP features across 62 false positives

This bar chart uses the **recorded Step 9 feature values**. Longer bars mean the feature had a larger average influence on the 62 false-positive predictions. Mean absolute SHAP shows the size of the influence, not its direction.

The code saves the same two formats from your original Step 9A: a **300-DPI PNG** and an **SVG**. It then displays the chart.

In [ ]:
# Recreate and save the original Step 9A feature-importance figure.
# Only the recorded Step 9 values are plotted; no new results are invented.
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(recorded_fp_top20['Feature'][::-1], recorded_fp_top20['Mean absolute SHAP'][::-1])
ax.set_xlabel('Mean absolute SHAP value (phishing probability)')
ax.set_title('Random Forest: Top 20 SHAP Features Across 62 False Positives')
fig.tight_layout()
fig.savefig('Random_Forest_Top20_SHAP_FP.png', dpi=300, bbox_inches='tight')
fig.savefig('Random_Forest_Top20_SHAP_FP.svg', bbox_inches='tight')
plt.show()
print('Saved: Random_Forest_Top20_SHAP_FP.png and Random_Forest_Top20_SHAP_FP.svg')


### Check the top-20 ranking against recalculated SHAP values

The next cell recomputes the table from the loaded model. It checks the first five recorded features. Small numerical differences may arise if the background or saved model differs. It does not overwrite the recorded table.

In [ ]:
feature_names_rf = np.asarray(tfidf_vectorizer.get_feature_names_out())
fp_mean_abs_rf = np.abs(fp_shap_values_rf).mean(axis=0)
fp_mean_signed_rf = fp_shap_values_rf.mean(axis=0)
ranked_rf = np.argsort(fp_mean_abs_rf)[::-1]
computed_fp_top20 = pd.DataFrame({
    'Feature': feature_names_rf[ranked_rf[:20]],
    'Mean absolute SHAP': fp_mean_abs_rf[ranked_rf[:20]],
    'Mean signed SHAP': fp_mean_signed_rf[ranked_rf[:20]],
})
display(computed_fp_top20)
assert list(computed_fp_top20['Feature'][:5]) == list(recorded_fp_top20['Feature'][:5]), (
    'The first five features do not match the recorded Step 9 ranking. Check the locked baseline and SHAP background.'
)

## 5. Which words pushed FP predictions in each direction? (Step 10)

A **positive signed SHAP value** moves the prediction toward phishing. A **negative signed SHAP value** moves it toward legitimate. This is different from the absolute importance ranking above.

### Recorded Step 10 — Selected strongest features

These are **examples from the original output**, not a newly calculated top-20 list.

In [ ]:
recorded_fp_toward_phishing = pd.DataFrame([
    {'Feature':'wrote','Mean signed SHAP':0.02333770},
    {'Feature':'org','Mean signed SHAP':0.02014730},
    {'Feature':'perl','Mean signed SHAP':0.00970759},
    {'Feature':'code','Mean signed SHAP':0.00757647},
    {'Feature':'mailing list','Mean signed SHAP':0.00732118},
])
recorded_fp_toward_legitimate = pd.DataFrame([
    {'Feature':'thanks','Mean signed SHAP':-0.00316245},
    {'Feature':'file','Mean signed SHAP':-0.00262233},
    {'Feature':'com www','Mean signed SHAP':-0.00245832},
    {'Feature':'release','Mean signed SHAP':-0.00211018},
])
print('Recorded features pushing FP predictions toward phishing:')
display(recorded_fp_toward_phishing.style.format({'Mean signed SHAP':'{:+.8f}'}))
print('Recorded features pushing FP predictions toward legitimate:')
display(recorded_fp_toward_legitimate.style.format({'Mean signed SHAP':'{:+.8f}'}))

**Explanation:** In these false positives, terms such as `wrote` and `org` often increased the phishing score. Terms such as `thanks` and `file` tended to reduce it. These are average SHAP contributions in the selected error group; they do **not** prove a word is always phishing-related.

The next cell computes the **full top-20 lists** from the actual SHAP matrix when it is available.

In [ ]:
signed_order_positive = np.argsort(fp_mean_signed_rf)[::-1]
signed_order_negative = np.argsort(fp_mean_signed_rf)
fp_toward_phishing = pd.DataFrame({'feature':feature_names_rf[signed_order_positive[:20]], 'mean_signed_shap':fp_mean_signed_rf[signed_order_positive[:20]]})
fp_toward_legitimate = pd.DataFrame({'feature':feature_names_rf[signed_order_negative[:20]], 'mean_signed_shap':fp_mean_signed_rf[signed_order_negative[:20]]})
print('Toward phishing:')
display(fp_toward_phishing)
print('Toward legitimate:')
display(fp_toward_legitimate)

## 6. Explain all 15 false negatives (Step 11)

A false negative is a phishing email wrongly predicted as legitimate. We use the **same baseline and background** to examine why these errors happened. No model is changed.

In [ ]:
fn_tfidf_rf = X_test_tfidf[false_negative_positions_rf]
fn_parts = []
for start in range(0, len(false_negative_positions_rf), 5):
    dense = fn_tfidf_rf[start:start+5].toarray()
    fn_parts.append(phishing_class_shap(explainer_rf.shap_values(dense, check_additivity=False)))
fn_shap_values_rf = np.vstack(fn_parts)
assert fn_shap_values_rf.shape == (15, 10000)
fn_reconstructed = expected_rf + fn_shap_values_rf.sum(axis=1)
fn_additivity_error = np.abs(fn_reconstructed - baseline_prob_rf[false_negative_positions_rf])
assert fn_additivity_error.max() < 1e-5
fn_mean_signed_rf = fn_shap_values_rf.mean(axis=0)
fn_toward_legitimate = pd.DataFrame({'feature':feature_names_rf[np.argsort(fn_mean_signed_rf)[:20]], 'mean_signed_shap':np.sort(fn_mean_signed_rf)[:20]})
fn_toward_phishing = pd.DataFrame({'feature':feature_names_rf[np.argsort(fn_mean_signed_rf)[::-1][:20]], 'mean_signed_shap':np.sort(fn_mean_signed_rf)[::-1][:20]})
print('FN maximum additivity error:', fn_additivity_error.max())
print('FN features toward legitimate:')
display(fn_toward_legitimate)
print('FN features toward phishing:')
display(fn_toward_phishing)

### Recorded Step 11 verification table

| Check | Original result |
|---|---:|
| False negatives explained | 15 |
| SHAP features | 10,000 |
| Maximum additivity error | 8.900361309072125e-09 |

The original output identified the following selected features. Negative values pushed these phishing emails toward the **legitimate** class.

In [ ]:
recorded_fn_toward_legitimate = pd.DataFrame([
    ('wrote',-0.08755146),('cheers',-0.02313851),('code',-0.01851887),
    ('samba',-0.01431133),('jonathan',-0.00999678),('lib',-0.00965811),
    ('perl',-0.00915535),('debian',-0.00881763),
], columns=['Feature','Mean signed SHAP'])
recorded_fn_toward_phishing = pd.DataFrame([
    ('org',0.00570359),('mailing list',0.00312099),
], columns=['Feature','Mean signed SHAP'])
print('Recorded FN features pushing toward legitimate:')
display(recorded_fn_toward_legitimate.style.format({'Mean signed SHAP':'{:+.8f}'}))
print('Recorded FN features pushing toward phishing:')
display(recorded_fn_toward_phishing.style.format({'Mean signed SHAP':'{:+.8f}'}))

**Explanation:** For the 15 missed phishing emails, `wrote` had a strong negative average contribution. This shows why **false-negative safety** matters: changing features to reduce false positives could also change phishing predictions. This notebook explains the errors but does **not** test an intervention.

## 7. Summary of RF 03

| Original step | Work completed | Main result |
|---|---|---|
| 6–7 | SHAP setup and TreeExplainer | SHAP 0.52.0; interventional background of 100 emails |
| 8 | SHAP for false positives | All 62 FPs analyzed |
| 9 | Feature importance ranking | 20 recorded FP features |
| 9A | Original SHAP importance bar chart | Top 20 FP features |
| 10 | Feature direction | Phishing- and legitimate-directed FP contributions |
| 11 | SHAP for false negatives | All 15 FNs analyzed |

**Main finding:** The locked Random Forest baseline made **62 false positives and 15 false negatives**. SHAP helped identify the features associated with these mistakes. These findings guide later experiments, but **no retraining, TF-IDF refitting, or performance improvement is claimed in RF 03**.

**Reproducibility note:** The recorded Markdown tables are visible on GitHub, and the plotting cell saves the original Step 9A figure when run. The computed SHAP tables require the original RF 02 checkpoint and Colab execution. This notebook does not contain stored model execution outputs.